# Shallow Diffuse 官方原始环境复现与 governed import 入口

该 Notebook 服务第二条链路: 官方原始环境复现 + governed import。它不把 legacy Stable Diffusion 结果混入 SD3.5 主表, 只生成补充表所需的官方参考记录、环境报告、命令日志、schema、validation report 和 Google Drive 压缩包。

正式逻辑位于 `paper_workflow/colab_utils/shallow_diffuse_official_reference.py`, Notebook 只负责 Colab 冷启动、参数注入、远程执行和打包。


## 运行边界

1. 默认样本数由 `SLM_WM_PAPER_RUN_SAMPLE_COUNT` 决定; `pilot_paper` 对应 600 个 prompt, `full_paper` 对应 6000 个 prompt, 用于 fixed-FPR=0.01 共同协议下的官方 legacy reference 复现。
2. Notebook 会在 helper 中按 `external_baseline/source_registry.json` 自动补齐 Shallow Diffuse 官方源码快照; 源码快照仍按外部缓存治理, 不进入 git 提交。
3. 该入口默认把 `SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_MODEL_ID` 设为 `Manojb/stable-diffusion-2-1-base`, 该路径是 `stabilityai/stable-diffusion-2-1-base` 不可直接访问时的公开镜像回退。
4. 该入口默认开启 `SLM_WM_SHALLOW_DIFFUSE_PATCH_MODEL_REPOSITORY_LAYOUT=1`, 会移除官方源码中对 `revision='fp16'` 分支的硬编码, 并通过环境变量收敛官方攻击器集合, 避免重型攻击器阻断 pilot_paper reference 复现。
5. 该入口默认开启 `SLM_WM_SHALLOW_DIFFUSE_PREPARE_LOCAL_MODEL_REPOSITORY=1`, 会把公开镜像下载到 `/content/shallow_diffuse_model_repository/stable_diffusion_2_1_base`, 并把本地 `model_index.json` 中的 `CLIPImageProcessor` 补丁为 legacy `transformers==4.23.1` 支持的 `CLIPFeatureExtractor`。
6. 该入口默认开启 `SLM_WM_SHALLOW_DIFFUSE_PREPARE_LEGACY_ENV=1`, 会在 `/content/shallow_diffuse_legacy_env` 中创建隔离 Python 3.9 legacy 环境, 并通过 `SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_PYTHON_EXECUTABLE` 间接调用官方脚本。
7. 默认攻击器集合为 `none`, 用于先证明官方 shallow latent injection、DDIM inversion 与 metric import 链路可运行。若后续需要攻击参考, 可把 `SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_ATTACKER_NAMES` 改为逗号分隔列表, 但重型 VAE / diffusion 攻击会显著增加依赖和运行时间。
8. 该入口默认显式传入官方 README 中的 `--reference_model ViT-g-14` 与 `--reference_model_pretrain laion2b_s12b_b42k`, 避免 OpenCLIP 自动选择不一致。


In [ ]:
SLM_WM_PAPER_RUN_NAME = "pilot_paper"

import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME


In [ ]:
%pip install -q --upgrade packaging huggingface_hub


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})

from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment
paper_run_environment = configure_paper_run_environment(
    workflow_name="official_reference_shallow_diffuse",
)
print(paper_run_environment)

from paper_workflow.colab_utils.dependency_check import build_notebook_dependency_report
dependency_report = build_notebook_dependency_report("official_reference_light")
print(dependency_report)


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
from pathlib import Path

source_dir = Path(os.environ['SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_SOURCE_DIR'])
entrypoint_path = source_dir / 'run_shallow_diffuse_t2i.py'
readme_path = source_dir / 'README.md'
print({
    'source_dir': str(source_dir),
    'source_dir_exists_before_helper': source_dir.exists(),
    'entrypoint_exists_before_helper': entrypoint_path.exists(),
    'readme_exists_before_helper': readme_path.exists(),
    'source_prepare_policy': 'helper_will_clone_from_external_baseline_source_registry_when_missing',
})
if readme_path.exists():
    print(readme_path.read_text(encoding='utf-8')[:4000])


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)


In [ ]:
from pathlib import Path

from paper_workflow.colab_utils.shallow_diffuse_official_reference import run_default_shallow_diffuse_official_reference_plan

shallow_diffuse_official_reference_summary = run_default_shallow_diffuse_official_reference_plan(root='.')
legacy_prepare_path = Path('outputs/shallow_diffuse_official_reference/shallow_diffuse_legacy_environment_prepare_result.json')
model_prepare_path = Path('outputs/shallow_diffuse_official_reference/shallow_diffuse_model_repository_prepare_result.json')
source_patch_path = Path('outputs/shallow_diffuse_official_reference/shallow_diffuse_official_source_patch_result.json')
for diagnostic_path in (legacy_prepare_path, model_prepare_path, source_patch_path):
    if diagnostic_path.exists():
        print(diagnostic_path)
        print(diagnostic_path.read_text(encoding='utf-8')[:6000])
shallow_diffuse_official_reference_summary


In [ ]:
from paper_workflow.colab_utils.notebook_entrypoint import package_workflow_outputs

archive_record = package_workflow_outputs(root='.', workflow_name="official_reference_shallow_diffuse")
archive_record.to_dict()


In [ ]:
import os
from pathlib import Path

for result_path in sorted(Path('outputs/shallow_diffuse_official_reference').rglob('*')):
    if result_path.is_file() and result_path.suffix.lower() in {'.json', '.jsonl', '.txt'}:
        print(result_path)
        print(result_path.read_text(encoding='utf-8', errors='ignore')[:4000])

expected_sample_count = int(os.environ['SLM_WM_PAPER_RUN_EXPECTED_SAMPLE_COUNT'])
assert shallow_diffuse_official_reference_summary['sample_count'] == expected_sample_count, shallow_diffuse_official_reference_summary
if shallow_diffuse_official_reference_summary['run_decision'] != 'pass':
    print({
        'shallow_diffuse_official_reference_ready': shallow_diffuse_official_reference_summary.get('shallow_diffuse_official_reference_ready'),
        'unsupported_reason': shallow_diffuse_official_reference_summary.get('unsupported_reason'),
        'legacy_environment_ready': shallow_diffuse_official_reference_summary.get('legacy_environment_ready'),
        'official_command_return_code': shallow_diffuse_official_reference_summary.get('official_command_return_code'),
    })
else:
    assert shallow_diffuse_official_reference_summary['reference_import_ready'] is True, shallow_diffuse_official_reference_summary
    assert shallow_diffuse_official_reference_summary['governed_reference_record_count'] > 0, shallow_diffuse_official_reference_summary
